In [ ]:
import pandas as pd
import numpy as np

#tf = pd.read_csv('../input/edgeiiotset-cyber-security-dataset-of-iot-iiot/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv', low_memory=False)
tf=pd.read_csv('./Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv', low_memory=False) 
#tf=pd.read_csv('./Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv', low_memory=False)

# Task 1: Model Training

In [ ]:
tf.info()
print(tf['Attack_type'].value_counts())
#['Normal','DDoS_UDP','DDoS_ICMP','DDoS_TCP','DDoS_HTTP']
df = tf.loc[tf['Attack_type'].isin( ['Normal','DDoS_UDP','DDoS_TCP','DDoS_HTTP'])]
print(df['Attack_type'].value_counts())
df.to_csv('DDoS_DNN.csv', encoding='utf-8', index=False)
df = pd.read_csv('./DDoS_DNN.csv', low_memory=False) 
#df
from sklearn.utils import shuffle
drop_columns = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4","arp.dst.proto_ipv4", 
                "http.file_data","http.request.full_uri","icmp.transmit_timestamp",
                "http.request.uri.query", "tcp.options","tcp.payload","tcp.srcport",
                "tcp.dstport", "udp.port", "mqtt.msg"]
drop_columns_related_other_attack=[ 'arp.opcode',
 'arp.hw.size',
 'icmp.checksum',
 'icmp.seq_le',
 'icmp.unused','dns.qry.name',
 'dns.qry.name.len',
 'dns.qry.qu',
 'dns.qry.type',
 'dns.retransmission',
 'dns.retransmit_request',
 'dns.retransmit_request_in',
 'mqtt.conack.flags',
 'mqtt.conflag.cleansess',
 'mqtt.conflags',
  "mqtt.hdrflags",
   "mqtt.len","mqtt.msg_decoded_as","mqtt.msgtype","mqtt.proto_len","mqtt.protoname","mqtt.topic","mqtt.topic_len",
                                   "mqtt.ver","mbtcp.len",'mbtcp.trans_id','mbtcp.unit_id']
df.drop(drop_columns, axis=1, inplace=True)
df.drop(drop_columns_related_other_attack, axis=1, inplace=True)
df.dropna(axis=0, how='any', inplace=True)
df.drop_duplicates(subset=None, keep="first", inplace=True)
df = shuffle(df)
df.isna().sum()
#print(df['Attack_type'].value_counts())


In [ ]:
############################model training#########################################

In [ ]:
feat_cols = list(df.columns)
label_col = "Attack_type"
feat_cols.remove(label_col)
# skip_list = ["icmp.unused", "http.tls_port", "dns.qry.type", "mqtt.msg_decoded_as"]
# df.drop(skip_list, axis=1, inplace=True)
# cat_names = {}
# for label_idx in cat_indices:
#     label = feature_names[label_idx]
#     print(label)
#     le = label_encoders[label]
#     le_name_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
#     le_value_mapping = dict(zip(le.transform(le.classes_), le.classes_))
#     print(le_value_mapping)
#     cat_names[label_idx] = le_value_mapping
from sklearn.preprocessing import LabelEncoder
categorical_index=[]
feat_cols = list(df.columns)
categorical_names = {}

categorical_features = ['http.request.method','http.referer',"http.request.version"]
categorical_index = [feat_cols.index(i) for i in categorical_features]
print(categorical_index)
# for i in categorical_features:
#     z=feat_cols.index(i)
#     print(z)
#     categorical_index.append(z)
df.info()
print("feat_col_len"+str(len(feat_cols)))
def Label_encoding(df, name,index):
    le = LabelEncoder()
    label = le.fit_transform(df[name])
   # df.drop(name, axis=1, inplace=True)
    df[name] = label
    le_value_mapping = dict(zip(le.transform(le.classes_), le.classes_))
    categorical_names[index] = le_value_mapping
    #categorical_names[index] = le.classes_
    print(name)
    print(le.classes_)

for i in categorical_features:
    z=feat_cols.index(i)
    Label_encoding(df,i,z)
print("after encoding \n")
df.info()


categorical_names
# feat_cols = list(df.columns)
# feat_cols.remove(label_col)


In [ ]:
feat_cols

In [ ]:
X = df.drop([label_col,'Attack_label'], axis=1)
y = df[label_col]

feat_cols = list(X.columns)

from sklearn.preprocessing import MaxAbsScaler 
categorical_features = ['http.request.method','http.referer',"http.request.version", "dns.qry.name.len","mqtt.conack.flags","mqtt.protoname","mqtt.topic"]
min_max_scaler=MaxAbsScaler()
for i in feat_cols:
    if i in categorical_features:
        continue
    else:
        X[[i]]=min_max_scaler.fit_transform(X[[i]])
      #  X_test[[i]]=min_max_scaler.transform(X_test[[i]])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)

#del X
#del y
print("X_train,y_train,X_test,y_test",len(X_train),len(y_train),len(X_test),len(y_test))
# 

In [ ]:
from sklearn.preprocessing import LabelEncoder

#encoder.fit(X)
#X_train= encoder.transform(X_train)
#X_test= encoder.transform(X_test)
label_encoder = LabelEncoder()
y_train =  label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)
label_encoder.classes_
print(y_test)
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(y_train),
                                                 y=y_train)

class_weights = {k: v for k,v in enumerate(class_weights)}
print(class_weights)




In [ ]:
# from sklearn.preprocessing import MaxAbsScaler 

# min_max_scaler = MaxAbsScaler()
# X_train =  min_max_scaler.fit_transform(X_train)
# X_test = min_max_scaler.transform(X_test)
# inver_Xtrain=min_max_scaler.inverse_transform(X_train)
# inver_Xtest=min_max_scaler.inverse_transform(X_test)
# feat_cols = list(X.columns)

# from sklearn.preprocessing import MaxAbsScaler 
# categorical_features = ['http.request.method','http.referer',"http.request.version", "dns.qry.name.len","mqtt.conack.flags","mqtt.protoname","mqtt.topic"]
# min_max_scaler=MaxAbsScaler()
# for i in feat_cols:
#     if i in categorical_features:
#         continue
#     else:
#         X_train[[i]]=min_max_scaler.fit_transform(X_train[[i]])
#         X_test[[i]]=min_max_scaler.transform(X_test[[i]])
X_train=X_train.to_numpy()
X_test=X_test.to_numpy()
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
input_shape = X_train.shape[1:]
print(X_train.shape, X_test.shape)
print(input_shape)



In [ ]:

print("X_train,X_test,feat_col",len(list(X_train[0])),len(list(X_test[0])),len(feat_cols))
X_train[0]
input_shape = X_train.shape[1:]

In [ ]:
num_classes = len(np.unique(y_train))
from  tensorflow.keras.utils import to_categorical 

y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)
print(y_train.shape, y_test.shape)

In [ ]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Input, ZeroPadding1D
from tensorflow.keras.layers import MaxPooling1D, Add, AveragePooling1D
from tensorflow.keras.layers import Dense, BatchNormalization, Activation
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Model
from keras.initializers import glorot_uniform
from tensorflow.keras.optimizers import Adam
import keras.backend as K
import tensorflow as tf

def f1_score(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    f1_val = 2*(precision*recall)/(precision+recall+K.epsilon())
    return f1_val
def identity_block(X, f, filters, stage, block):
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'
    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv1D(filters=F1, kernel_size=1, strides=1, padding='valid', name=conv_name_base + '2a', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv1D(filters=F2, kernel_size=f, strides=1, padding='same', name=conv_name_base + '2b', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv1D(filters=F3, kernel_size=1, strides=1, padding='valid', name=conv_name_base + '2c', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2c')(X)

    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X
def convolutional_block(X, f, filters, stage, block, s=2):
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    F1, F2, F3 = filters

    X_shortcut = X

    X = Conv1D(filters=F1, kernel_size=1, strides=s, padding='valid', name=conv_name_base + '2a', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2a')(X)
    X = Activation('relu')(X)

    X = Conv1D(filters=F2, kernel_size=f, strides=1, padding='same', name=conv_name_base + '2b', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2b')(X)
    X = Activation('relu')(X)

    X = Conv1D(filters=F3, kernel_size=1, strides=1, padding='valid', name=conv_name_base + '2c', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name=bn_name_base + '2c')(X)

    X_shortcut = Conv1D(filters=F3, kernel_size=1, strides=s, padding='valid', name=conv_name_base + '1', kernel_initializer=glorot_uniform(seed=0))(X_shortcut)
    X_shortcut = BatchNormalization(name=bn_name_base + '1')(X_shortcut)

    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)

    return X
def ResNet50(input_shape):
    X_input = Input(input_shape)
    X = ZeroPadding1D((3, 3))(X_input)

    X = Conv1D(filters=64, kernel_size=7, strides=2, name='conv1', kernel_initializer=glorot_uniform(seed=0))(X)
    X = BatchNormalization(name='bn_conv1')(X)
    X = Activation('relu')(X)
    X = MaxPooling1D(pool_size=3, strides=2)(X)

    X = convolutional_block(X, f=3, filters=[64, 64, 256], stage=2, block='a', s=1)
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='b')
    X = identity_block(X, 3, [64, 64, 256], stage=2, block='c')


    X = convolutional_block(X, f=3, filters=[128, 128, 512], stage=3, block='a', s=2)
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='b')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='c')
    X = identity_block(X, 3, [128, 128, 512], stage=3, block='d')

    X = convolutional_block(X, f=3, filters=[256, 256, 1024], stage=4, block='a', s=2)
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='b')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='c')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='d')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='e')
    X = identity_block(X, 3, [256, 256, 1024], stage=4, block='f')

    X = X = convolutional_block(X, f=3, filters=[512, 512, 2048], stage=5, block='a', s=2)
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='b')
    X = identity_block(X, 3, [512, 512, 2048], stage=5, block='c')

    X = AveragePooling1D(pool_size=2, padding='same')(X)
    model = Model(inputs=X_input, outputs=X, name='ResNet50')

    return model
from keras.metrics import Recall, Precision

def build_model(num_classes, input_shape=(92, 1)):
    base_model = ResNet50(input_shape=input_shape)
    headModel = base_model.output
    headModel = Flatten()(headModel)
    headModel=Dense(256, activation='relu', name='fc1',kernel_initializer=glorot_uniform(seed=0))(headModel)
    headModel=Dense(128, activation='relu', name='fc2',kernel_initializer=glorot_uniform(seed=0))(headModel)
    headModel = Dense(num_classes, activation='softmax', name='fc3',kernel_initializer=glorot_uniform(seed=0))(headModel)
    model = Model(inputs=base_model.input, outputs=headModel)
    opt = Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss= tf.keras.metrics.categorical_crossentropy, 
                  metrics=['accuracy', Recall(), Precision(), f1_score])
    return model
#num_classes = len(np.unique(y_train))
model = build_model(num_classes, input_shape=input_shape)
model.summary()

In [ ]:
# from tensorflow.keras.utils import plot_model

# plot_model(model, to_file="model_fig.jpg", show_shapes=True)
# from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# #from livelossplot import PlotLossesKeras


# model_weights_file_path = "model.h5"
# checkpoint = ModelCheckpoint(filepath=model_weights_file_path, monitor="val_loss", verbose=1, save_best_only=True, mode="min", save_weights_only=True)
# early_stopping = EarlyStopping(monitor="val_loss", mode="min", verbose=1, patience=10)
# lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, mode="min", verbose=1, min_lr=0)
# #plotlosses = PlotLossesKeras()

# # call_backs = [checkpoint, early_stopping, lr_reduce, plotlosses]
# call_backs = [checkpoint, early_stopping, lr_reduce]
# EPOCHS = 50
# BATCH_SIZE = 256
# #history=model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE)
# history = model.fit(X_train, y_train, 
#                     validation_data=(X_test, y_test),
#                     #validation_split=0.1,
#                     epochs=EPOCHS, 
#                     batch_size=BATCH_SIZE,
#                     callbacks=call_backs,
#                     class_weight=class_weights,
#                     verbose=1)

In [ ]:
#model.load_weights("/kaggle/input/model-for-ddos/model.h5")
#model.load_weights("model_ddos_http_udp_tcp.h5")
# # Assuming model expects [batch_size, time_steps, features]
# y = model.predict(X_test[1].reshape(1, X_test[1].shape[0], X_test[1].shape[1]))

# y
# #model.load_weights("model_ML_dataset.h5")

# Task2:Process Flow in Cloud Server

In [ ]:
#ReceivedNetwokTrafficData  PreprocessNetworkTrafficData  FeedNetworkTrafficDataToDLModel  ExecuteDLModel GetDLModelPrediction GenerateAttackStatus& ConsequenceResult 

In [ ]:
def PreprocessNetworkTrafficData(X):
    return X
    

In [ ]:
#execute DLModel
model.load_weights("model_ddos_http_udp_tcp_v2.h5")

In [ ]:
def GetMLModelPrediction(X):
    
    #Retrieve list of training features names from classifier
#     cols_when_model_builds = loaded_model.feature_names_in_
#     print("cols_when_model_builds",cols_when_model_builds)
#     X = X[cols_when_model_builds]
   
    y=model.predict(X.reshape(1, X_test[1].shape[0], X_test[1].shape[1]))
    print("Model prediction for y",y)
    return y
# GetMLModelPrediction(model,X_test[1].reshape(1, X_test[1].shape[0], X_test[1].shape[1]))
# y_test[9]

In [ ]:
import lime
from lime import lime_tabular
label_encoder.classes_
# # LIME has one explainer for all the models
#X_train1 = X_train.reshape(X_train.shape[0], X_train.shape[1])
#explainer = lime.lime_tabular.LimeTabularExplainer(X_train1, feature_names=feat_cols,feature_selection='lasso_path',
 #                                                 class_names=label_encoder.classes_)
X_train1 = X_train.reshape(X_train.shape[0], X_train.shape[1])
X_test1 = X_test.reshape(X_test.shape[0], X_test.shape[1])
explainer = lime.lime_tabular.LimeTabularExplainer(X_train1, feature_names=feat_cols,feature_selection='lasso_path',
                                                  class_names=label_encoder.classes_,categorical_features=categorical_index, 
                                                  categorical_names=categorical_names)
categorical_features = ['http.request.method',
                        'http.referer',"http.request.version",
                        "dns.qry.name.len","mqtt.conack.flags","mqtt.protoname","mqtt.topic"]
class_names=label_encoder.classes_
# for i in categorical_features:
i=105
import random
from numpy import argmax

y_test1=argmax(y_test,axis=1)
y_train1=argmax(y_train, axis=1)
print("True Attack Name: "+class_names[y_test1[i]]+"\n")
#exp = explainer.explain_instance(inver_Xtest[i], model.predict,num_features=92,top_labels=1)
exp = explainer.explain_instance(X_test1[i], model.predict,num_features=10,top_labels=1)
exp.show_in_notebook(show_all=False)

In [ ]:
def lime_explanation(test):
    
    exp = explainer.explain_instance(test, model.predict,num_features=10,top_labels=2)
    exp.show_in_notebook(show_all=True)
    instance_id=random.randint(0,10000)#random id
    
    #     file_to_save="Condition_changed_instance num "+str(i)+".csv"
    class_names=label_encoder.classes_
  
    file_to_save="LimeExplanation_"+str(instance_id)+".csv"
#     with open(file_to_save, 'w') as fp:
#            # fp.write("True Attack Name: "+class_names[y_test[i]]+"\n")
#             fp.write("Attack_type,"+"Probabilistic Value,"+"Feature Name,"+"Expected_range,"+"Feature_Weight,"+"Feature_Value"+"\n")
#             fp.close()
#             available_labels=exp.available_labels()
#             for j in available_labels:
#                 List=exp.as_list(j)
#                 Lime_instance_explanation(exp,test,List,j,file_to_save)
    with open(file_to_save, 'w') as fp:
           # fp.write("True Attack Name: "+class_names[y_test[i]]+"\n")
            fp.write("Attack_type,"+"Probabilistic Value,"+"Feature Name,"+"Expected_range,"+"Feature_Weight,"+"Feature_Value"+"\n")
            fp.close()
            available_labels=exp.available_labels()
            #for j in available_labels:
            List=exp.as_list(available_labels[0])
            classname=class_names[available_labels[0]]
            html_file=classname+'_'+str(instance_id)+'.html'
            exp.save_to_file(html_file)
            Lime_instance_explanation(exp,test,List,available_labels[0],file_to_save)
    ##Json file generation
    print("lime_Explanation done")
    return file_to_save
    

In [ ]:
def Lime_instance_explanation(exp, X_test1, mylist, class_id, file_name):
    class_names = label_encoder.classes_
    file_to_save = file_name
    #instance_id=random.randint(0,10000)#random id
    instance_id = file_name.replace("LimeExplanation_", "").replace(".csv", "")
    print(instance_id)
    with open(file_to_save, 'a') as fp:
        for i in mylist:
            mystring=i
            feature=mystring[0]
            condition=mystring[0]
            feature_weight=mystring[1]
            splitted=feature.split(" ")
            length=len(splitted)
            arith_exp=['<','<=','>','>=','=>']
            #expected_range='x'+feature
            #print("Lime_instance_explanation",length)
            if length<=3:
                #print("True3")
                max_len=0
                feature_id=""
                for i in range(len(splitted)):
                    if len(splitted[i])>max_len:
                        max_len=len(splitted[i])
                        feature_id=i
                feature_name=splitted[feature_id]
#                 #print(feature_name)
#                 expected_range=""
#                 splitted[feature_id]=feature_name
#                 for i in splitted:
#                     expected_range=expected_range+i
            else:
                if splitted[1] in arith_exp:
                    to_replace=splitted[0]+' '+splitted[1]
                    #expected_range=to_replace+expected_range
                    feature=feature.replace(to_replace,'')
                if splitted[length-2] in arith_exp:
                    #print("True")
                    to_replace=splitted[length-2]+' '+splitted[length-1]
                    #expected_range=expected_range+to_replace
                    feature=feature.replace(to_replace,'')
                feature_name=feature.strip()
                #print(feature_name)
            predict_value=exp.predict_proba[class_id]
            
            categorical=feature_name.split("=")
            #print("categorical",categorical)
            if len(categorical)==2:
                categorical_name=categorical[0]
                categorical_value=categorical[1]
                fp.write(str(instance_id)+","+class_names[class_id]+","+str(predict_value)+","+categorical_name+","+condition+","+str(feature_weight)+","+categorical_value+"\n")

            else:
                col_name=feature_name
                #print("Lime_instance_explanation",col_name)
                z=feat_cols.index(col_name)
                #print("Lime_instance_explanation",z)
                #print("Lime_instance_explanation X_text1",X_test1)
                #print("Lime_instance_explanation X_text1 featurevalue",X_test1[z])
                Feature_Value=X_test1[z]
                fp.write(str(instance_id)+","+class_names[class_id]+","+str(predict_value)+","+feature_name+","+condition+","+str(feature_weight)+","+str(Feature_Value)+"\n")
            
            #print("fp.write done")


#     with open(file_to_save, 'a') as fp:
#         for i in mylist:
#             mystring = i
#             feature = mystring[0]
#             condition = mystring[0]
#             feature_weight = mystring[1]
#             splitted = feature.split(" ")
#             length = len(splitted)
#             arith_exp = ['<', '<=', '>', '>=', '=>']
#             expected_range = 'x' + feature
            
#             if length <= 3:
#                 max_len = 0
#                 feature_id = 0
#                 for idx, word in enumerate(splitted):
#                     if len(word) > max_len:
#                         max_len = len(word)
#                         feature_id = idx
#                 feature_name = splitted[feature_id]
#                 expected_range = " ".join(splitted)
#             else:
#                 if splitted[1] in arith_exp:
#                     to_replace = ' '.join(splitted[:2])
#                     expected_range = to_replace + expected_range
#                     feature = feature.replace(to_replace, '')
#                 if splitted[length-2] in arith_exp:
#                     to_replace = ' '.join(splitted[-2:])
#                     expected_range = expected_range + to_replace
#                     feature = feature.replace(to_replace, '')
#                 feature_name = feature.strip()
            
#             # Assuming exp.predict_proba[class_id] gives the prediction probability
#             predict_value = exp.predict_proba[class_id] if hasattr(exp, 'predict_proba') else exp[class_id]
            
#             # Feature value extraction with additional checks
#             col_name = feature_name.strip()
#             try:
#                 z = feat_cols.index(col_name)
                
#                 # Debugging prints
#                 #print(f"X_test1[0]: {X_test1[0]}")
#                 #print(f"Type of X_test1[0]: {type(X_test1[0])}")
#                 #print(f"z: {z}")
                
#                 # Check if X_test1[0] is indexable (array-like)
#                 if isinstance(X_test1[0], (list, np.ndarray)):
#                     Feature_Value = X_test1[0][z]  # Access feature value if it's an array
#                 else:
#                     Feature_Value = X_test1[0]  # Use scalar directly
#             except ValueError:
#                 print(f"Feature {col_name} not found in feat_cols.")
#                 Feature_Value = "N/A"
            
#             # Writing to file
#             fp.write(f"{class_names[class_id]},{predict_value},{feature_name},{condition},{feature_weight},{Feature_Value}\n")


In [ ]:
# def Lime_instance_explanation(exp,X_test1,mylist, class_id,file_name):
#     class_names=label_encoder.classes_
#     #file_to_save="instance num "+str(i)+".csv"
#     file_to_save=file_name
    
#     with open(file_to_save, 'a') as fp:
#         for i in mylist:
#             mystring=i
#             feature=mystring[0]
#             condition=mystring[0]
#             feature_weight=mystring[1]
#             splitted=feature.split(" ")
#             length=len(splitted)
#             arith_exp=['<','<=','>','>=','=>']
#             expected_range='x'+feature
#             print(length)
#             if length<=3:
#                 print("True3")
#                 max_len=0
#                 feature_id=""
#                 for i in range(len(splitted)):
#                     if len(splitted[i])>max_len:
#                         max_len=len(splitted[i])
#                         feature_id=i
#                 feature_name=splitted[feature_id]
#                 print(feature_name)
#                 expected_range=""
#                 splitted[feature_id]=feature_name
#                 for i in splitted:
#                     expected_range=expected_range+i
#             else:
#                 if splitted[1] in arith_exp:
#                     to_replace=splitted[0]+' '+splitted[1]
#                     expected_range=to_replace+expected_range
#                     feature=feature.replace(to_replace,'')
#                 if splitted[length-2] in arith_exp:
#                     print("True")
#                     to_replace=splitted[length-2]+' '+splitted[length-1]
#                     expected_range=expected_range+to_replace
#                     feature=feature.replace(to_replace,'')
#                 feature_name=feature.strip()
#                 print(feature_name)
#             predict_value=exp.predict_proba[class_id]
#             col_name=feature_name
#             z=feat_cols.index(col_name)
#             Feature_Value=X_test1[0][z]
#             fp.write(class_names[class_id]+","+str(predict_value)+","+feature_name+","+condition+","+str(feature_weight)+","+str(Feature_Value)+"\n")


#  Task2: SGenerateAttackStatus& ConsequenceResult

In [ ]:
# Function to find all paths for each goal
def add_paths_to_goals(gsn_df, paths_df):
    # Create a new column 'Matching Paths' to store the matching paths for each goal
    gsn_df["Matching Paths"] = ""

    # Iterate over each goal in the GSN table
    for idx, gsn_row in gsn_df.iterrows():
        matching_paths = []

        # Check all paths to see if the current goal is present
        for _, path_row in paths_df.iterrows():
            path = path_row["Paths"]
            if gsn_row["Goal"] in path:
                matching_paths.append(path)

        # Store the matching paths as a string (joined by ';')
        gsn_df.at[idx, "Matching Paths"] = "; ".join(matching_paths)

    return gsn_df

In [ ]:
# Function to generate interpretation info from DataFrame group
def generate_interpretation_info_from_group(group):
    interpretation_info = []
    Goal_info=[]
    for _, row in group.iterrows():
        info = {
            "goal": row["Goal"],
            "goal_Name":row['Goal Name'],
            "details":
            {
                
                "process": row["Proc_Name"],
                "process Importance":row["Degree"],
                "attribute": row["Feature Name"],
                "state": row["Expected_range"],
                "attribute value":row["Feature_Value"], 
                "Path":row["Matching Paths"]
            }
            
        }
        interpretation_info.append(info)
    return interpretation_info


In [ ]:
def create_solution_goal_pair_rows(df_input):
    """
    Transforms the DataFrame to create a distinct row for each
    (Solution, Immediate Goal) pair, with the 'Matching Paths' column
    updated to show only the specific path relevant to that pair.

    Args:
        df_input (pd.DataFrame): DataFrame containing at least 'Solution' and
                                 'Matching Paths' columns. Other columns will be
                                 duplicated in the output.

    Returns:
        pd.DataFrame: A new DataFrame where rows from df_input are exploded
                      for each unique (immediate goal, specific path) found
                      for a solution. Includes 'Immediate Goal' and updates
                      'Matching Paths' to the specific path. Rows from the
                      input that do not yield any (Solution, Immediate Goal)
                      pairs are excluded.
    """
    if df_input.empty:
        # If input is empty, return an empty DataFrame with expected columns
        # 'Immediate Goal' is new, 'Matching Paths' content would be specific.
        output_cols = df_input.columns.tolist()
        if "Immediate Goal" not in output_cols:
            output_cols.append("Immediate Goal")
        return pd.DataFrame(columns=output_cols)

    df = df_input.copy()
    # This temporary column will store a list of (immediate_goal, specific_path) tuples
    temp_goal_path_pairs_col = "_internal_goal_path_pairs"

    # Helper function to apply to each row
    def get_all_immediate_goal_path_pairs_for_row(row_series):
        solution_node = str(row_series["Solution"])
        # This 'Matching Paths' is the semicolon-separated string of ALL paths for the solution
        paths_string = row_series["Matching Paths"] 
        
        goal_path_pairs = [] # To store (goal, specific_path) tuples

        if pd.isna(paths_string) or not paths_string.strip():
            return [] # Return an empty list if no paths information

        individual_paths = paths_string.split(';')
        for path_str_raw in individual_paths:
            path_str = path_str_raw.strip()
            if not path_str:
                continue
            
            elements = path_str.split('|')
            # Check if path is long enough and ends with the solution node
            if len(elements) >= 2 and elements[-1] == solution_node:
                potential_immediate_goal = elements[-2]
                # Check if the preceding element is a goal
                if potential_immediate_goal.startswith('G'):
                    # Add the (goal, specific path that led to it) tuple
                    goal_path_pairs.append((potential_immediate_goal, path_str))
        
        # Return a list of unique (goal, path) tuples for this solution row
        # If multiple distinct paths lead to the same goal, each will be a separate entry.
        # If the same path is listed multiple times leading to the same goal, it will be unique here.
        return list(set(goal_path_pairs))

    # 1. Create a temporary column with the list of (immediate_goal, specific_path) pairs
    df[temp_goal_path_pairs_col] = df.apply(get_all_immediate_goal_path_pairs_for_row, axis=1)

    # 2. Filter out rows that didn't yield any (goal, path) pairs
    df_with_pairs = df[df[temp_goal_path_pairs_col].map(len) > 0].copy()

    if df_with_pairs.empty:
        # If no rows have any valid pairs after processing,
        # return an empty DataFrame with the correct columns
        output_cols = df_input.columns.tolist()
        if "Immediate Goal" not in output_cols:
            output_cols.append("Immediate Goal")
        return pd.DataFrame(columns=output_cols)

    # 3. Explode the DataFrame based on the list of (goal, path) pairs
    df_exploded = df_with_pairs.explode(temp_goal_path_pairs_col)

    # 4. Populate 'Immediate Goal' from the first element of the pair
    #    and overwrite 'Matching Paths' with the second element (the specific path)
    #    Handle cases where a row might become all NaN after explode if a list was empty (though filtered)
    df_exploded_valid = df_exploded.dropna(subset=[temp_goal_path_pairs_col]).copy()


    df_exploded_valid["Immediate Goal"] = df_exploded_valid[temp_goal_path_pairs_col].apply(lambda x: x[0])
    # Overwrite the 'Matching Paths' column with the specific path for this goal
    df_exploded_valid["Matching Paths"] = df_exploded_valid[temp_goal_path_pairs_col].apply(lambda x: x[1])
    
    # 5. Define final columns: all original columns (with 'Matching Paths' now updated)
    #    plus the new 'Immediate Goal' column. Drop the temporary processing column.
    final_columns = df_input.columns.tolist()
    if "Immediate Goal" not in final_columns:
        final_columns.append("Immediate Goal") # Ensure 'Immediate Goal' is in the list
    
    df_output = df_exploded_valid[final_columns].copy()
    df_output = df_output.reset_index(drop=True)
#print(df_reset)
    return df_output


In [ ]:
# (It's good practice for this function to work on a copy if it modifies the input)
def add_paths_to_goals(gsn_df_orig, paths_df):
    gsn_df = gsn_df_orig.copy()
    gsn_df["Matching Paths"] = ""
    for idx, gsn_row in gsn_df.iterrows():
        matching_paths_list = []
        current_solution_node = str(gsn_row["Solution"])
        for _, path_row in paths_df.iterrows():
            path_str = path_row["Paths"]
            # It's more robust to check if current_solution_node is an element
            # rather than just a substring, e.g., if path_str.split('|') contains current_solution_node
            if current_solution_node in path_str.split('|'): # Assuming solution is a distinct node
                matching_paths_list.append(path_str)
                #print("Solution, ",current_solution_node,path_str)
        gsn_df.loc[idx, "Matching Paths"] = "; ".join(matching_paths_list)
    return gsn_df


In [ ]:
def extract_intermediate_strategies_goals(df_input):
    """
    Extracts intermediate strategies, a set of unique intermediate goals,
    and ordered series of intermediate goals from paths in a DataFrame.
    'SI-' prefixed nodes (not the target goal) are considered intermediate goals.

    Args:
        df_input (pd.DataFrame): DataFrame with an 'Immediate Goal' column (which defines
                                 the target goal for path segment analysis) and a
                                 'Matching Paths' column. 'Matching Paths' should
                                 contain semicolon-separated path strings, where each
                                 path is a pipe-separated sequence of nodes.

    Returns:
        pd.DataFrame: The input DataFrame with three new columns:
                      'Intermediate Strategies' (comma-separated string of unique strategies).
                      'Intermediate Goals' (comma-separated string of unique intermediate goals,
                                           including G- and SI- prefixed nodes).
                      'Ordered Intermediate Goal Series' (semicolon-separated strings;
                                                       each string is a pipe-separated,
                                                       ordered list of intermediate goals
                                                       from a qualifying path segment).
    """
    df = df_input.copy()

    df["Intermediate Strategies"] = ""
    df["Intermediate Goals"] = ""
    df["Ordered Intermediate Goal Series"] = "" # New column

    start_marker_node = "SI-4 Req"

    for idx, row in df.iterrows():
        current_target_goal = str(row["Immediate Goal"]) # Ensure string
        paths_string = row["Matching Paths"]
        #print("current_target_goal",current_target_goal, paths_string)
        row_intermediate_strategies = set()
        row_unique_intermediate_goals = set()
        row_ordered_goal_series_strings = set()

        if pd.isna(paths_string) or not paths_string.strip():
            continue

        individual_paths = paths_string.split(';')

        for path_str_raw in individual_paths:
            path_str = path_str_raw.strip()
            if not path_str:
                continue

            elements = path_str.split('|')

            try:
                start_node_idx = -1
            except ValueError:
                continue 

            try:
                offset = start_node_idx + 1
                if offset >= len(elements):
                    continue
                relative_end_node_idx = elements[offset:].index(current_target_goal)
                end_node_idx = relative_end_node_idx + offset
            except ValueError:
                continue 
            
            if start_node_idx >= end_node_idx -1: 
                continue 
            
            intermediate_elements = elements[start_node_idx + 1 : end_node_idx]
            
            current_path_ordered_goals = [] 

            for el in intermediate_elements:
                # 1. Populate Intermediate Strategies set
                if el.startswith('S') and not el.startswith('Sn') and not el.startswith('SI-'):
                    row_intermediate_strategies.add(el)
                
                # 2. Check if 'el' is an intermediate goal node
                is_intermediate_goal_node = False
                # An element is an intermediate goal if it starts with 'G' or 'SI-'
                # AND it is not the current_target_goal for the segment.
                if el.startswith('G')  and el != current_target_goal:
                    is_intermediate_goal_node = True
                elif el.startswith('SI-') and el != current_target_goal:
                    is_intermediate_goal_node = True
                if is_intermediate_goal_node:
                    row_unique_intermediate_goals.add(el)
                    current_path_ordered_goals.append(el)

            if current_path_ordered_goals:
                row_ordered_goal_series_strings.add("|".join(current_path_ordered_goals))
        #print("current_target_goal row_intermediate_strategies",current_target_goal,row_intermediate_strategies,row_unique_intermediate_goals,row_ordered_goal_series_strings)
        df.loc[idx, "Intermediate Strategies"] = ", ".join(sorted(list(row_intermediate_strategies)))
        df.loc[idx, "Intermediate Goals"] = ", ".join(sorted(list(row_unique_intermediate_goals)))
        df.loc[idx, "Ordered Intermediate Goal Series"] = "; ".join(sorted(list(row_ordered_goal_series_strings)))
        #print(df.loc[idx, ['Matching Paths', 'Immediate Goal', 'Intermediate Goals']])
    return df


In [ ]:
import pandas as pd
# Assuming node_desc_map and process_importance_map are pre-loaded as in your script

def generate_interpretation_info_from_group(smap_group_data, node_desc_map, process_importance_map):
    temp_df_for_grouping = smap_group_data.copy()
    # Ensure 'Immediate Goal' is string for grouping and lookup, handle NaNs
    temp_df_for_grouping['__grouping_key_goal_id__'] = temp_df_for_grouping['Immediate Goal'].fillna('__MISSING_ID__').astype(str).str.strip()
    
    grouped_by_immediate_goal = temp_df_for_grouping.groupby('__grouping_key_goal_id__')
    final_interpretation_list = []

    for goal_id_for_group, rows_for_this_goal in grouped_by_immediate_goal:
        if goal_id_for_group == '__MISSING_ID__':
            # print(f"Warning: Skipping S_map rows with missing 'Immediate Goal' ID.") # Optional: for debugging
            continue

        goal_name_for_group = node_desc_map.get(goal_id_for_group, f"Name not found for ID '{goal_id_for_group}'")
        
        # --- Collections for aggregated and unique data ---
        unique_attribute_tuples = set()
        collected_ordered_series_set = set()
        collected_strategy_ids_set = set()
        collected_intermediate_goal_ids_set = set()

        # Iterate over all S_map rows for this specific goal_id
        for _, s_map_row in rows_for_this_goal.iterrows():
            # 1. Collect data for unique attribute contexts
            attr_tuple = (
                s_map_row.get("Feature Name", "N/A"),
                s_map_row.get("Expected_range", "N/A"),
                s_map_row.get("Feature_Value", "N/A")
            )
            unique_attribute_tuples.add(attr_tuple)

            # 2. Collect data for aggregated "affected goal" series
            ogs_str = s_map_row.get("Ordered Intermediate Goal Series", "")
            if pd.notna(ogs_str) and ogs_str.strip():
                collected_ordered_series_set.add(ogs_str.strip())

            # 3. Collect data for aggregated strategies
            strategies_str = s_map_row.get("Intermediate Strategies", "")
            if pd.notna(strategies_str) and strategies_str.strip():
                strategy_ids = [s.strip() for s in strategies_str.split(',') if s.strip()]
                collected_strategy_ids_set.update(strategy_ids)
            
            # 4. Collect data for aggregated intermediate goals
            intermediate_goals_str = s_map_row.get("Intermediate Goals", "")
            if pd.notna(intermediate_goals_str) and intermediate_goals_str.strip():
                intermediate_goal_ids = [g.strip() for g in intermediate_goals_str.split(',') if g.strip()]
                collected_intermediate_goal_ids_set.update(intermediate_goal_ids)

        # --- Process collected data for the output object ---

        # A. Finalize "affected goal" string (from Ordered Intermediate Goal Series)
        # Sort for consistent output order if multiple unique series are joined
        affected_goal_output_str = "; ".join(sorted(list(collected_ordered_series_set))) if collected_ordered_series_set else "Path Not Available"

        # B. Finalize "process" payload (from Intermediate Strategies)
        strategies_payload_list = []
        for strat_id in sorted(list(collected_strategy_ids_set)): # Sort for consistent output
            strategy_description = node_desc_map.get(strat_id, f"Name not found for ID '{strat_id}'")
            extracted_process_name = "N/A"

            # Only attempt to extract process name if strategy_description is not a "not found" message
            if not strategy_description.startswith("Name not found for ID"):
                try:
                    words = strategy_description.split()
                    if len(words) > 2:
                        extracted_process_name = words[2]
                    else:
                        extracted_process_name = "FormatError (short desc)"
                except Exception:
                    extracted_process_name = "ErrorExtractingProcess"
            
            process_importance_val = process_importance_map.get(extracted_process_name, "Importance Not Found")
            
            # Skip if the original strategy ID wasn't found, or process extraction failed, or importance is missing
            if strategy_description.startswith("Name not found for ID") or \
               extracted_process_name in ["N/A", "FormatError (short desc)", "ErrorExtractingProcess"] or \
               process_importance_val == "Importance Not Found":
                # This filter mirrors the original intent: if extracted_process_name.startswith("Name not found") or process_importance_val=="Importance Not Found"
                # The original check `extracted_process_name.startswith("Name not found")` would only trigger if `words[2]` yielded "Name"
                # and `strategy_description` was very specific like "Name not found process...".
                # The check `strategy_description.startswith("Name not found for ID")` is more direct for missing strategy descriptions.
                # The original line: `if extracted_process_name.startswith("Name not found")or process_importance_val=="Importance Not Found" : continue`
                # This implies that even if extracted_process_name was derived, if it itself signaled "Name not found" (unlikely) or if importance was missing, skip.
                # The refined logic above should be more robust:
                #   1. Skip if `strat_id` has no description.
                #   2. Skip if `extracted_process_name` is an error/placeholder.
                #   3. Skip if `process_importance_val` is "Importance Not Found".
                # The user's filter `if extracted_process_name.startswith("Name not found") or process_importance_val=="Importance Not Found" :`
                # Let's use a filter closer to original line 65 intent, but more robust for `extracted_process_name`:
                should_skip = False
                if process_importance_val == "Importance Not Found":
                    should_skip = True
                # Check if extracted_process_name indicates it couldn't be properly determined
                if extracted_process_name in ["N/A", "FormatError (short desc)", "ErrorExtractingProcess"]:
                     should_skip = True
                # The original `extracted_process_name.startswith("Name not found")` is tricky.
                # If strategy_description was "Name not found for ID X", then `extracted_process_name` would be "N/A" with the above logic.
                # The only way `extracted_process_name` itself could be "Name not found.." is if `words[2]` produced it from a valid description, which is not the intent.
                # So, the primary conditions are: valid strategy_description -> valid extracted_process_name -> valid process_importance_val.

                if strategy_description.startswith("Name not found for ID") or process_importance_val == "Importance Not Found":
                     # This is a simpler and more direct interpretation of the likely original skip conditions
                    continue
            
            strategies_payload_list.append({
                "process_name": extracted_process_name,
                "process_importance": process_importance_val
            })
        
        # C. Finalize "goal description" payload (from Intermediate Goals)
        intermediate_goals_payload_list = []
        for inter_goal_id in sorted(list(collected_intermediate_goal_ids_set)): # Sort for consistent output
            goal_name = node_desc_map.get(inter_goal_id, f"Name not found for ID '{inter_goal_id}'")
            # Optionally, filter out goals where name is "Name not found..." if desired
            # if not goal_name.startswith("Name not found for ID"):
            intermediate_goals_payload_list.append({
                "goal_id": inter_goal_id,
                "goal_name": goal_name
            })

        # D. Finalize unique "attribute_contexts" list
        attribute_contexts_list_final = []
        # Sort tuples for consistent output order before converting to dicts
        # Sorting by attribute, then state, then value (all cast to str for robust comparison)
        sorted_attribute_tuples = sorted(
            list(unique_attribute_tuples), 
            key=lambda t: (str(t[0]), str(t[1]), str(t[2]))
        )
        for t_attr, t_state, t_val in sorted_attribute_tuples:
            attribute_contexts_list_final.append({
                "attribute": t_attr,
                "state": t_state,
                "attribute_value": t_val
            })
            
        # Construct the goal-centric object
        goal_centric_object = {
            "goal_id": goal_id_for_group,
            "goal_name": goal_name_for_group,
            "attribute_contexts": attribute_contexts_list_final,
            "affected goal": affected_goal_output_str,
            "goal description": intermediate_goals_payload_list, # Corresponds to user's "goal" key
            "process": strategies_payload_list                 # Corresponds to user's "strategy" key
        }
        final_interpretation_list.append(goal_centric_object)
        
    return final_interpretation_list

In [ ]:
import json
import io # For file handling flexibility if needed

# --- Helper function to load GSN node data and create a name lookup map ---
def load_and_prepare_gsn_nodes(file_path):
    """
    Reads the GSN node data (pipe-delimited) and creates a lookup map
    from Node_Name to Description.
    Assumes 'Node_Name' and 'Description' are columns in the CSV.
    """
    try:
        # Use low_memory=False if there are mixed types, though may not be needed for this file
        df_nodes = pd.read_csv(file_path, sep='|', low_memory=False)

        # Ensure required columns exist
        if 'Node_Name' not in df_nodes.columns or 'Description' not in df_nodes.columns:
            raise ValueError(f"CSV file {file_path} must contain 'Node_Name' and 'Description' columns. Found: {df_nodes.columns.tolist()}")

        # Clean data: drop rows where Node_Name or Description is NaN, strip whitespace
        df_cleaned = df_nodes.dropna(subset=['Node_Name', 'Description']).copy()
        df_cleaned.loc[:, 'Node_Name'] = df_cleaned['Node_Name'].astype(str).str.strip()
        df_cleaned.loc[:, 'Description'] = df_cleaned['Description'].astype(str).str.strip()

        # Create map: if duplicate Node_Name exists, keep the first description
        node_map_df = df_cleaned.drop_duplicates(subset=['Node_Name'], keep='first')
        node_name_map = pd.Series(node_map_df['Description'].values, index=node_map_df['Node_Name']).to_dict()
        
        return node_name_map
    except Exception as e:
        #print(f"Error loading or processing GSN node file '{file_path}': {e}")
        return {} # Return an empty map on error

# --- Helper function to load Process Importance data ---
def load_process_importance(file_path, separator=None): # Pass separator if not comma, e.g., '\s+' for spaces
    try:
        if separator:
            df_importance = pd.read_csv(file_path, sep=separator, low_memory=False)
        else: # Assumes comma-separated or pandas can infer
            df_importance = pd.read_csv(file_path, low_memory=False)

        if 'Process_Name' not in df_importance.columns or 'Degree' not in df_importance.columns:
            raise ValueError(f"Process importance file ('{file_path}') must contain 'Process_Name' and 'Degree' columns. Found: {df_importance.columns.tolist()}")

        df_cleaned = df_importance.dropna(subset=['Process_Name', 'Degree']).copy()
        df_cleaned.loc[:, 'Process_Name'] = df_cleaned['Process_Name'].astype(str).str.strip()
        # Assuming 'Degree' is the importance value, ensure it's suitable for JSON (e.g., numeric or string)
        # df_cleaned.loc[:, 'Degree'] = pd.to_numeric(df_cleaned['Degree'], errors='coerce') # Optional: convert to numeric

        map_df = df_cleaned.drop_duplicates(subset=['Process_Name'], keep='first')
        process_importance_map = pd.Series(map_df['Degree'].values, index=map_df['Process_Name']).to_dict()
        #print(f"Successfully loaded process importance. Map size: {len(process_importance_map)}")
        return process_importance_map
    except Exception as e:
        #print(f"Error loading process importance from '{file_path}': {e}")
        return {}

In [ ]:
def ProtectionControlConfiguration(Lime_file):
    #extract goal attribute relation
    file_name='.\\Network_Traffic_Packet_Record_v1.csv'
    Network_Traffic_record=pd.read_csv(file_name, low_memory=False) 
   # Network_Traffic_record
    #extract all_possible path
    file_name='.\All_possible_path.txt'
    All_possible_path=pd.read_csv(file_name, low_memory=False) 
    #All_possible_path
    #All_possible_path.info()
    # --- Step 1: Generate gsn_df_with_paths ---
    gsn_df_with_paths = add_paths_to_goals(Network_Traffic_record, All_possible_path)
    # print("--- DataFrame with Matching Paths ---")
    # print(gsn_df_with_paths)
    # print("\n")

    # --- Step 2: Create the Solution-Goal pair rows ---
    df_solution_goal_pairs = create_solution_goal_pair_rows(gsn_df_with_paths)

    # --- Display the result ---
    #print("--- DataFrame with Solution-Goal Pairs (Exploded) ---")
    #print(df_solution_goal_pairs[['Solution', 'Solution Name', 'Atrribute', 'Immediate Goal', 'Matching Paths']])
    gsn_df_with_paths=extract_intermediate_strategies_goals(df_solution_goal_pairs)
    #file_name='./DDoS_UDP ddos attack for 38155.csv'
    exp=pd.read_csv(Lime_file, low_memory=False) 
    #exp=pd.read_csv(file_name, low_memory=False) 
    #exp.info()
    exp.sort_values(by=['Feature_Weight'],ascending=False)
    # Filter out negative Feature_Weight values
    exp = exp[exp['Feature_Weight'] >= 0]
    # Merging the tables on Feature Name (from exp) and Attribute (from S_sac)
    merged_df = pd.merge(exp, gsn_df_with_paths, left_on='Feature Name', right_on='Atrribute', how='inner')

    # Display the merged dataframe
    #print(merged_df)
    #cols=[1,2,3,4,5,6,7,8]

    S_map=merged_df#[merged_df.columns[cols]]
    # 1. Load GSN node descriptions (Node ID -> Description map)
    gsn_nodes_file = '.\\GSNIOTDDOS2_node.csv' # Ensure this file path is correct
    node_descriptions = load_and_prepare_gsn_nodes(gsn_nodes_file)

    # 2. Load Process Importance (Process Name -> Degree map)
    process_importance_file = '.\\weighted_pagerank.txt' # Ensure this file path is correct
    # You might need to specify a separator for weighted_pagerank.txt, e.g., sep='\s+' if space-separated
    # If it's comma-separated, pandas might infer it or you can use sep=','
    process_importances = load_process_importance(process_importance_file)
    if not S_map.empty and node_descriptions and process_importances:
        grouped = S_map.groupby(['Attack_type', 'Probabilistic Value'])
        structured_data = []

        s_map_instance_id = S_map['Instance_id'].iloc[0] if 'Instance_id' in S_map.columns and not S_map.empty else "Default_ID"

        for (attack, probability), group_data in grouped:
            traffic_info = {
                "Type": 'attack',
                "Spec_version": "dummy",
                "Id": str(s_map_instance_id),
                "Created": "N/A",
                "modified": "N/A",
                "Name": attack,
                "probability": float(probability)
            }

            interpretation_info_list = generate_interpretation_info_from_group(
                group_data, node_descriptions, process_importances
            )

            structured_entry = {
                "traffic_info": traffic_info,
                "interpretation_info": interpretation_info_list
            }
            structured_data.append(structured_entry)

        json_data = json.dumps(structured_data, indent=4)
        #print(json_data)
        instance_id = Lime_file.replace("LimeExplanation_", "").replace(".csv", "")
        
        file_name="structured_data_enriched"+str(instance_id)+"_.json"
        with open(file_name, "w") as file:
            file.write(json_data)
    #else:
        #print("Skipping JSON generation due to errors in loading S_map or lookup files.")
    #Send_email_To_Stackholder("structured_data.json")


In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
def Send_email_To_Stackholder(File_name):
    # Email credentials
    sender_email = "tejpata15@gmail.com"
    receiver_email = 'masrufabayesh@gmail.com'
    password = "dhvj tysv yhtr mtxx" # Note: For Gmail, you may need to use an App Password

    # Create the MIMEMultipart message
    msg = MIMEMultipart()
    msg['From'] = sender_email
    msg['To'] = receiver_email
    msg['Subject'] = 'DDOS_ATTACK'

    # Email body
    body = 'Please find the attached file.'
    msg.attach(MIMEText(body, 'plain'))

    # Specify the file to attach
    filename = File_name  # Path to your file
    print("send email to stackholder: message sending")
    # Open the file in binary mode and attach it to the email
    with open(filename, 'rb') as attachment:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(attachment.read())
        encoders.encode_base64(part)  # Encode the file in base64

        part.add_header(
            'Content-Disposition',
            f'attachment; filename={filename}',  # File name as it will appear in the email
        )

        # Attach the file to the message
        msg.attach(part)

    # Gmail SMTP server details
    smtp_server = 'smtp.gmail.com'
    port = 587  # For TLS

    # Sending the email
    try:
        server = smtplib.SMTP(smtp_server, port)
        server.starttls()  # Start TLS for security
        server.login(sender_email, password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        print("Email with attachment sent successfully!")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        server.quit()


In [ ]:
# import socket
# import json
# import random
# import time
# import csv
# import os
# import numpy as np
# def ReceivedNetwokTrafficData(ip, port):#ReceivedSensorData
#     # Create a TCP/IP socket
#     server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

#     # Bind the socket to the address and port
#     server_socket.bind((ip, port))

#     # Listen for incoming connections
#     server_socket.listen(5)
#     print(f"TCP server listening on {ip}:{port}")
#     # Save execution time to CSV file
#     attack_types = ['DDoS_HTTP', 'DDoS_TCP', 'DDoS_UDP', 'Normal']
#     log_filename = "attack_execution_time_log.csv"
#     # Decode y_test to get readable labels
#     decoded_y_test = [attack_types[argmax(row)] for row in y_test]

#     # Create indices for only DDoS_TCP and DDoS_UDP
#     valid_attack_labels = [ 'DDoS_UDP']
#     attack_indices = [i for i, label in enumerate(decoded_y_test) if label in valid_attack_labels]
#     while True:
#         # Wait for a connection
#         client_socket, client_address = server_socket.accept()
#         #print(f"Connection from {client_address} established.")

#         try:
#             start_time = time.time()  # ⏱️ Start timing
#             # Receive the data in small chunks
#             data = b""
#             while True:
#                 chunk = client_socket.recv(1024)
#                 if not chunk:
#                     break
#                 data += chunk

#             # Decode and process the received data
#             data = data.decode('utf-8')
# #             
#             #print("Received mean DataFrame:")
# #             print(mean_data)
        
#             # Convert string to dictionary
#             data_dict = json.loads(data)
#             # Convert dictionary to pandas DataFrame
#             ReceivedNetwokTrafficData = pd.DataFrame(data_dict)
#             X_test1 = X_test.reshape(X_test.shape[0], X_test.shape[1])
#             #index = random.randrange(0, len(X_test1))  # Adjust range to fit your DataFrame size
#             index = random.choice(attack_indices)
#             ReceivedNetwokTrafficData=X_test1[index]
#             #ReceivedNetwokTrafficData.info()
#             #print("ReceivedNetwokTrafficData")
# #             #FeedSensorDataToMLModel
#             FeedNetworkDataToMLModel=PreprocessNetworkTrafficData(ReceivedNetwokTrafficData)
#             #print("PreprocessNetworkTrafficData done")
# #           #ExecuteMLModel
#             #MLModel=model
# #             #GetMLModelPrediction
#             #print("MLModel print")
#             modelprediction=GetMLModelPrediction(FeedNetworkDataToMLModel)#PreprocessedSensorData
#             #print("modelprediction done")
        
#             lime_explanation_file=lime_explanation(FeedNetworkDataToMLModel)
#             ProtectionControlConfiguration(lime_explanation_file)
# #             #Send_To_Application(modelprediction)
# #             DecidetoTriggerActuator(modelprediction)
#     # 📤 Send report to stakeholder
#             # This function is called by lime_explanation, so it’s indirectly covered
#             # Send_email_To_Stackholder("structured_data.json")  <-- already inside called function

#             end_time = time.time()
#             total_time = end_time - start_time
            
#             # 🧠 Get predicted attack label
#             predicted_idx = np.argmax(modelprediction)
#             predicted_attack = attack_types[predicted_idx]

#             #print(f"\n🚀 Total Pipeline Execution Time for {predicted_attack}: {total_time:.4f} seconds")

#             # 📝 Save result to CSV
#             file_exists = os.path.isfile(log_filename)
#             with open(log_filename, "a", newline="") as csvfile:
#                 writer = csv.writer(csvfile)
#                 if not file_exists:
#                     writer.writerow(["Attack_Type", "Execution_Time"])  # Write header
#                 writer.writerow([predicted_attack, f"{total_time:.6f}"])


           
           
            


#         except Exception as e:
#             print(f"Error handling client {client_address}: {e}")


#         finally:
#             # Close the connection
#             client_socket.close()

In [ ]:
import socket
import json
import random
import time
import csv
import os
import numpy as np
import pandas as pd  # Required for DataFrame

def ReceivedNetwokTrafficData(ip, port):
    server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_socket.bind((ip, port))
    server_socket.listen(5)
    print(f"TCP server listening on {ip}:{port}")

    attack_types = ['DDoS_HTTP', 'DDoS_TCP', 'DDoS_UDP', 'Normal']
    
    log_filename = "attack_execution_time_log.csv"

    decoded_y_test = [attack_types[np.argmax(row)] for row in y_test]
    valid_attack_labels = ['DDoS_UDP']
    attack_indices = [i for i, label in enumerate(decoded_y_test) if label in valid_attack_labels]

    while True:
        client_socket, client_address = server_socket.accept()

        try:
            start_time = time.time()

            data = b""
            while True:
                chunk = client_socket.recv(1024)
                if not chunk:
                    break
                data += chunk

            data = data.decode('utf-8')
            data_dict = json.loads(data)
            ReceivedNetwokTrafficData = pd.DataFrame(data_dict)

            X_test1 = X_test.reshape(X_test.shape[0], X_test.shape[1])
            index = random.choice(attack_indices)
            ReceivedNetwokTrafficData = X_test1[index]

            FeedNetworkDataToMLModel = PreprocessNetworkTrafficData(ReceivedNetwokTrafficData)

            modelprediction = GetMLModelPrediction(FeedNetworkDataToMLModel)
            t1 = time.time()  # After model prediction

            lime_explanation_file = lime_explanation(FeedNetworkDataToMLModel)
            t2 = time.time()  # After explanation

            ProtectionControlConfiguration(lime_explanation_file)
            t3 = time.time()  # After control config

            # Calculate times
            model_prediction_time = t1 - start_time
            explanation_time = t2 - t1
            control_to_end_time = t3 - t2
            total_time = t3 - start_time

            predicted_idx = np.argmax(modelprediction)
            predicted_attack = attack_types[predicted_idx]

            # Save to CSV
            file_exists = os.path.isfile(log_filename)
            with open(log_filename, "a", newline="") as csvfile:
                writer = csv.writer(csvfile)
                if not file_exists:
                    writer.writerow([
                        "Attack_Type",
                        "Model_Prediction_Time",
                        "Explanation_Time",
                        "Control_to_End_Time",
                        "Total_Time"
                    ])
                writer.writerow([
                    predicted_attack,
                    f"{model_prediction_time:.6f}",
                    f"{explanation_time:.6f}",
                    f"{control_to_end_time:.6f}",
                    f"{total_time:.6f}"
                ])

        except Exception as e:
            print(f"Error handling client {client_address}: {e}")
        finally:
            client_socket.close()


In [ ]:
import socket
import threading
# Main function to start the cloud server
def main():
    # Set the IP and port for receiving data from the edge
    hostname = socket.gethostname()
    tcp_ip = socket.gethostbyname(hostname)
    tcp_ip='0.0.0.0'
    tcp_receive_port = 12349  # Port to receive data from edge server

    # Start the cloud server to listen for data from the edge
    receive_thread = threading.Thread(target=ReceivedNetwokTrafficData, args=(tcp_ip, tcp_receive_port), daemon=True)
    receive_thread.start()

    # Keep the main thread alive
    receive_thread.join()

if __name__ == '__main__':
    main()